In [2]:

import re

with open('/workspace/mpl2026/src/data/mpl2026.ts', 'r') as f:
    raw = f.read()

# Parser les tournois
pattern = re.compile(
    r"\{\s*id:\s*'([^']+)',\s*name:\s*'([^']+)',\s*club_id:\s*'([^']+)',\s*club_name:\s*'([^']+)',\s*date:\s*'([^']+)',\s*region:\s*'([^']+)',\s*category:\s*'([^']+)',\s*division:\s*'([^']+)',\s*type:\s*'([^']+)',\s*status:\s*'([^']+)'[^}]*max_teams:\s*(\d+)\s*\}"
)

rows = pattern.findall(raw)
print(f"Tournois parsés: {len(rows)}")

# Vérifier les catégories uniques
cats = sorted(set(r[6] for r in rows))
print(f"Catégories: {cats}")
divs = sorted(set(r[7] for r in rows))
print(f"Divisions: {divs}")
statuses = sorted(set(r[9] for r in rows))
print(f"Statuts: {statuses}")


Tournois parsés: 279
Catégories: ['M100', 'M1000', 'M25', 'M250', 'M50', 'M500', 'MIXED', 'U10', 'U12', 'U14']
Divisions: ['junior', 'men', 'mixed']
Statuts: ['completed', 'open', 'upcoming']


In [5]:

import re

with open('/workspace/mpl2026/src/data/mpl2026.ts', 'r') as f:
    raw = f.read()

pattern = re.compile(
    r"\{\s*id:\s*'([^']+)',\s*name:\s*'([^']+)',\s*club_id:\s*'([^']+)',\s*club_name:\s*'([^']+)',\s*date:\s*'([^']+)',\s*region:\s*'([^']+)',\s*category:\s*'([^']+)',\s*division:\s*'([^']+)',\s*type:\s*'([^']+)',\s*status:\s*'([^']+)'[^}]*max_teams:\s*(\d+)\s*\}"
)
rows = pattern.findall(raw)

sql = """\
-- ═══════════════════════════════════════════════════════════════════════════
-- MPL 2026 — Script SQL Supabase
-- Table : tournaments
-- 279 tournois officiels · Saison 10 Jan 2026 → 26 Déc 2026
-- Généré automatiquement depuis mpl2026.ts
-- ═══════════════════════════════════════════════════════════════════════════

-- ── 1. CRÉATION DE LA TABLE ───────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS public.tournaments (
  id               TEXT        PRIMARY KEY,
  name             TEXT        NOT NULL,
  club_id          TEXT        NOT NULL,
  club_name        TEXT        NOT NULL,
  date             DATE        NOT NULL,
  region           TEXT        NOT NULL
                               CHECK (region IN ('Nord', 'Ouest', 'Centre', 'Est', 'Sud')),
  category         TEXT        NOT NULL
                               CHECK (category IN ('M25','M50','M100','M250','M500','M1000','MIXED','U10','U12','U14')),
  division         TEXT        NOT NULL
                               CHECK (division IN ('men','women','junior','mixed')),
  type             TEXT        NOT NULL,
  status           TEXT        NOT NULL DEFAULT 'upcoming'
                               CHECK (status IN ('upcoming','open','completed','cancelled')),
  max_teams        INTEGER     NOT NULL DEFAULT 16,
  teams_registered INTEGER     NOT NULL DEFAULT 0,
  created_at       TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  updated_at       TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

-- ── 2. INDEX ──────────────────────────────────────────────────────────────

CREATE INDEX IF NOT EXISTS idx_tournaments_date     ON public.tournaments (date);
CREATE INDEX IF NOT EXISTS idx_tournaments_region   ON public.tournaments (region);
CREATE INDEX IF NOT EXISTS idx_tournaments_category ON public.tournaments (category);
CREATE INDEX IF NOT EXISTS idx_tournaments_status   ON public.tournaments (status);
CREATE INDEX IF NOT EXISTS idx_tournaments_division ON public.tournaments (division);
CREATE INDEX IF NOT EXISTS idx_tournaments_club_id  ON public.tournaments (club_id);

-- ── 3. ROW LEVEL SECURITY ─────────────────────────────────────────────────

ALTER TABLE public.tournaments ENABLE ROW LEVEL SECURITY;

-- Lecture publique pour tous
CREATE POLICY "tournaments_select_public"
  ON public.tournaments FOR SELECT
  USING (true);

-- Écriture réservée aux admins authentifiés
CREATE POLICY "tournaments_insert_admin"
  ON public.tournaments FOR INSERT
  WITH CHECK (auth.role() = 'authenticated');

CREATE POLICY "tournaments_update_admin"
  ON public.tournaments FOR UPDATE
  USING (auth.role() = 'authenticated');

CREATE POLICY "tournaments_delete_admin"
  ON public.tournaments FOR DELETE
  USING (auth.role() = 'authenticated');

-- ── 4. TRIGGER updated_at ─────────────────────────────────────────────────

CREATE OR REPLACE FUNCTION public.handle_updated_at()
RETURNS TRIGGER AS $$
BEGIN
  NEW.updated_at = NOW();
  RETURN NEW;
END;
$$ LANGUAGE plpgsql;

DROP TRIGGER IF EXISTS trg_tournaments_updated_at ON public.tournaments;
CREATE TRIGGER trg_tournaments_updated_at
  BEFORE UPDATE ON public.tournaments
  FOR EACH ROW EXECUTE FUNCTION public.handle_updated_at();

-- ── 5. INSERT DES DONNÉES ─────────────────────────────────────────────────
-- Utilise ON CONFLICT pour permettre un ré-import sans erreur

INSERT INTO public.tournaments
  (id, name, club_id, club_name, date, region, category, division, type, status, max_teams, teams_registered)
VALUES
"""

escape = lambda s: s.replace("'", "''")

lines = []
for (tid, name, club_id, club_name, date, region, cat, div, typ, status, max_t) in rows:
    lines.append(
        f"  ('{escape(tid)}', '{escape(name)}', '{escape(club_id)}', '{escape(club_name)}', "
        f"'{date}', '{escape(region)}', '{escape(cat)}', '{escape(div)}', "
        f"'{escape(typ)}', '{escape(status)}', {max_t}, 0)"
    )

sql += ',\n'.join(lines)
sql += """

ON CONFLICT (id) DO UPDATE SET
  name             = EXCLUDED.name,
  club_id          = EXCLUDED.club_id,
  club_name        = EXCLUDED.club_name,
  date             = EXCLUDED.date,
  region           = EXCLUDED.region,
  category         = EXCLUDED.category,
  division         = EXCLUDED.division,
  type             = EXCLUDED.type,
  status           = EXCLUDED.status,
  max_teams        = EXCLUDED.max_teams,
  updated_at       = NOW();

-- ── 6. VÉRIFICATION RAPIDE ───────────────────────────────────────────────

SELECT
  category,
  COUNT(*)          AS total,
  COUNT(*) FILTER (WHERE status = 'completed')  AS terminés,
  COUNT(*) FILTER (WHERE status = 'open')       AS ouverts,
  COUNT(*) FILTER (WHERE status = 'upcoming')   AS à_venir
FROM public.tournaments
GROUP BY category
ORDER BY category;

-- ── FIN DU SCRIPT ─────────────────────────────────────────────────────────
-- Total attendu : 279 tournois
"""

with open('/workspace/mpl2026_tournaments.sql', 'w') as f:
    f.write(sql)

print(f"✅ Fichier SQL généré : mpl2026_tournaments.sql")
print(f"   Lignes totales : {len(sql.splitlines())}")
print(f"   Tournois insérés : {len(rows)}")
print(f"\nRépartition par catégorie :")
from collections import Counter
cats = Counter(r[6] for r in rows)
for cat, count in sorted(cats.items()):
    print(f"   {cat:8s}  {count:3d}")


✅ Fichier SQL généré : mpl2026_tournaments.sql
   Lignes totales : 389
   Tournois insérés : 279

Répartition par catégorie :
   M100       52
   M1000       4
   M25        53
   M250       51
   M50        53
   M500       10
   MIXED      14
   U10        14
   U12        14
   U14        14


In [8]:

import re
from collections import Counter

with open('/workspace/mpl2026/src/data/mpl2026.ts', 'r') as f:
    raw = f.read()

pattern = re.compile(
    r"\{\s*id:\s*'([^']+)',\s*name:\s*'([^']+)',\s*club_id:\s*'([^']+)',\s*club_name:\s*'([^']+)',\s*date:\s*'([^']+)',\s*region:\s*'([^']+)',\s*category:\s*'([^']+)',\s*division:\s*'([^']+)',\s*type:\s*'([^']+)',\s*status:\s*'([^']+)'[^}]*max_teams:\s*(\d+)\s*\}"
)
rows = pattern.findall(raw)
print(f"Tournois parsés : {len(rows)}")

escape = lambda s: s.replace("'", "''")

sql = """\
-- ═══════════════════════════════════════════════════════════════════════════
-- MPL 2026 — Script SQL Supabase
-- Table : tournaments
-- 279 tournois officiels · Saison 10 Jan 2026 → 26 Déc 2026
-- Colonnes sans mots réservés PostgreSQL
-- ═══════════════════════════════════════════════════════════════════════════

-- ── 1. SUPPRESSION DE L'ANCIENNE TABLE (si elle existe) ───────────────────

DROP TABLE IF EXISTS public.tournaments CASCADE;

-- ── 2. CRÉATION DE LA TABLE ───────────────────────────────────────────────

CREATE TABLE public.tournaments (
  id                TEXT        PRIMARY KEY,
  name              TEXT        NOT NULL,
  club_id           TEXT        NOT NULL,
  club_name         TEXT        NOT NULL,
  tournament_date   DATE        NOT NULL,
  region            TEXT        NOT NULL
                                CHECK (region IN ('Nord','Ouest','Centre','Est','Sud')),
  category          TEXT        NOT NULL
                                CHECK (category IN ('M25','M50','M100','M250','M500','M1000','MIXED','U10','U12','U14')),
  division          TEXT        NOT NULL
                                CHECK (division IN ('men','women','junior','mixed')),
  tournament_type   TEXT        NOT NULL,
  status            TEXT        NOT NULL DEFAULT 'upcoming'
                                CHECK (status IN ('upcoming','open','completed','cancelled')),
  max_teams         INTEGER     NOT NULL DEFAULT 16,
  teams_registered  INTEGER     NOT NULL DEFAULT 0,
  created_at        TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  updated_at        TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

-- ── 3. INDEX ──────────────────────────────────────────────────────────────

CREATE INDEX idx_tournaments_date     ON public.tournaments (tournament_date);
CREATE INDEX idx_tournaments_region   ON public.tournaments (region);
CREATE INDEX idx_tournaments_category ON public.tournaments (category);
CREATE INDEX idx_tournaments_status   ON public.tournaments (status);
CREATE INDEX idx_tournaments_division ON public.tournaments (division);
CREATE INDEX idx_tournaments_club_id  ON public.tournaments (club_id);

-- ── 4. ROW LEVEL SECURITY ─────────────────────────────────────────────────

ALTER TABLE public.tournaments ENABLE ROW LEVEL SECURITY;

CREATE POLICY "tournaments_select_public"
  ON public.tournaments FOR SELECT
  USING (true);

CREATE POLICY "tournaments_insert_admin"
  ON public.tournaments FOR INSERT
  WITH CHECK (auth.role() = 'authenticated');

CREATE POLICY "tournaments_update_admin"
  ON public.tournaments FOR UPDATE
  USING (auth.role() = 'authenticated');

CREATE POLICY "tournaments_delete_admin"
  ON public.tournaments FOR DELETE
  USING (auth.role() = 'authenticated');

-- ── 5. TRIGGER updated_at ─────────────────────────────────────────────────

CREATE OR REPLACE FUNCTION public.handle_updated_at()
RETURNS TRIGGER AS $$
BEGIN
  NEW.updated_at = NOW();
  RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_tournaments_updated_at
  BEFORE UPDATE ON public.tournaments
  FOR EACH ROW EXECUTE FUNCTION public.handle_updated_at();

-- ── 6. INSERT DES DONNÉES (279 tournois) ─────────────────────────────────

INSERT INTO public.tournaments
  (id, name, club_id, club_name, tournament_date, region, category,
   division, tournament_type, status, max_teams, teams_registered)
VALUES
"""

lines = []
for (tid, name, club_id, club_name, date, region, cat, div, typ, status, max_t) in rows:
    lines.append(
        f"  ('{escape(tid)}', '{escape(name)}', '{escape(club_id)}', '{escape(club_name)}', "
        f"'{date}', '{escape(region)}', '{escape(cat)}', '{escape(div)}', "
        f"'{escape(typ)}', '{escape(status)}', {max_t}, 0)"
    )

sql += ',\n'.join(lines)
sql += """

ON CONFLICT (id) DO UPDATE SET
  name             = EXCLUDED.name,
  club_id          = EXCLUDED.club_id,
  club_name        = EXCLUDED.club_name,
  tournament_date  = EXCLUDED.tournament_date,
  region           = EXCLUDED.region,
  category         = EXCLUDED.category,
  division         = EXCLUDED.division,
  tournament_type  = EXCLUDED.tournament_type,
  status           = EXCLUDED.status,
  max_teams        = EXCLUDED.max_teams,
  updated_at       = NOW();

-- ── 7. VÉRIFICATION ──────────────────────────────────────────────────────

SELECT
  category,
  COUNT(*)                                              AS total,
  COUNT(*) FILTER (WHERE status = 'completed')          AS termines,
  COUNT(*) FILTER (WHERE status = 'open')               AS ouverts,
  COUNT(*) FILTER (WHERE status = 'upcoming')           AS a_venir
FROM public.tournaments
GROUP BY category
ORDER BY category;

-- Résumé global
SELECT COUNT(*) AS total_tournois FROM public.tournaments;

-- ── FIN DU SCRIPT ─────────────────────────────────────────────────────────
-- Total attendu : 279 tournois
-- Catégories   : M25(53) M50(53) M100(52) M250(51) M500(10) M1000(4)
--                MIXED(14) U10(14) U12(14) U14(14)
"""

with open('/workspace/mpl2026_tournaments.sql', 'w') as f:
    f.write(sql)

print(f"✅ SQL régénéré")
print(f"   Lignes : {len(sql.splitlines())}")
print(f"   Tournois : {len(rows)}")
cats = Counter(r[6] for r in rows)
for cat, cnt in sorted(cats.items()):
    print(f"   {cat:8s} × {cnt}")


Tournois parsés : 279
✅ SQL régénéré
   Lignes : 395
   Tournois : 279
   M100     × 52
   M1000    × 4
   M25      × 53
   M250     × 51
   M50      × 53
   M500     × 10
   MIXED    × 14
   U10      × 14
   U12      × 14
   U14      × 14


In [11]:

import re

with open('/workspace/mpl2026/src/data/mpl2026.ts', 'r') as f:
    content = f.read()

# Compter avant
before = content.count("MEN&WOMEN")
print(f"MEN&WOMEN avant : {before}")

# Pour chaque ligne MEN&WOMEN, on génère 2 lignes :
# - une MEN avec division 'men', id original + 'h' (hommes)
# - une WOMEN avec division 'women', id original + 'f' (femmes)

def split_mw(m):
    line = m.group(0)

    id_m    = re.search(r"id: '([^']+)'", line)
    name_m  = re.search(r"name: '([^']+)'", line)
    cid_m   = re.search(r"club_id: '([^']+)'", line)
    cname_m = re.search(r"club_name: '([^']+)'", line)
    date_m  = re.search(r"date: '([^']+)'", line)
    reg_m   = re.search(r"region: '([^']+)'", line)
    cat_m   = re.search(r"category: '([^']+)'", line)
    stat_m  = re.search(r"status: '([^']+)'", line)
    mt_m    = re.search(r"max_teams: (\d+)", line)

    if not all([id_m, name_m, cid_m, cname_m, date_m, reg_m, cat_m, stat_m, mt_m]):
        return line  # sécurité

    oid   = id_m.group(1)
    name  = name_m.group(1)
    cid   = cid_m.group(1)
    cn    = cname_m.group(1)
    date  = date_m.group(1)
    reg   = reg_m.group(1)
    cat   = cat_m.group(1)
    stat  = stat_m.group(1)
    mt    = mt_m.group(1)

    men_line = (
        f"  {{ id: '{oid}h', name: '{name} (Hommes)', club_id: '{cid}', club_name: '{cn}', "
        f"date: '{date}', region: '{reg}', category: '{cat}', division: 'men', "
        f"type: 'MEN', status: '{stat}', max_teams: {mt} }},"
    )
    women_line = (
        f"  {{ id: '{oid}f', name: '{name} (Dames)', club_id: '{cid}', club_name: '{cn}', "
        f"date: '{date}', region: '{reg}', category: '{cat}', division: 'women', "
        f"type: 'WOMEN', status: '{stat}', max_teams: {mt} }},"
    )
    return men_line + '\n' + women_line

# Pattern : ligne complète avec MEN&WOMEN
pattern = re.compile(r"  \{[^}]+type: 'MEN&WOMEN'[^}]+\},")
new_content = pattern.sub(split_mw, content)

# Vérifications
after_mw  = new_content.count("MEN&WOMEN")
men_count  = len(re.findall(r"type: 'MEN'", new_content))
women_count = len(re.findall(r"type: 'WOMEN'", new_content))
junior_count = len(re.findall(r"type: 'JUNIOR'", new_content))
mixed_count  = len(re.findall(r"type: 'MIXED'", new_content))
pure_men = len(re.findall(r"(?<!')MEN(?!&|'h|'f)", new_content))

print(f"MEN&WOMEN restants : {after_mw}  (attendu 0)")
print(f"type MEN   : {men_count}")
print(f"type WOMEN : {women_count}")
print(f"type JUNIOR: {junior_count}")
print(f"type MIXED : {mixed_count}")

# Compter toutes les entrées de tournois
all_entries = re.findall(r"id: '[^']+'", new_content)
# Exclure les entrées de clubs (club_id, id dans Club objects)
tourn_entries = re.findall(r"\{ id: '[^']+'.*?type: '(?:MEN|WOMEN|JUNIOR|MIXED)'", new_content)
print(f"\nTotal entrées tournois (type connu) : {len(tourn_entries)}")
print(f"  = {before} originaux dont {before} MEN&WOMEN × 2 = {before - before + len(tourn_entries)} total attendu ~{279-before + before*2}")

with open('/workspace/mpl2026/src/data/mpl2026.ts', 'w') as f:
    f.write(new_content)

print("\n✅ mpl2026.ts mis à jour")


MEN&WOMEN avant : 157
MEN&WOMEN restants : 1  (attendu 0)
type MEN   : 222
type WOMEN : 156
type JUNIOR: 42
type MIXED : 14

Total entrées tournois (type connu) : 434
  = 157 originaux dont 157 MEN&WOMEN × 2 = 434 total attendu ~436

✅ mpl2026.ts mis à jour


In [14]:

import re
from collections import Counter

with open('/workspace/mpl2026/src/data/mpl2026.ts', 'r') as f:
    raw = f.read()

# Parser tous les tournois
pattern = re.compile(
    r"\{\s*id:\s*'([^']+)',\s*name:\s*'([^']+)',\s*club_id:\s*'([^']+)',\s*club_name:\s*'([^']+)',\s*"
    r"date:\s*'([^']+)',\s*region:\s*'([^']+)',\s*category:\s*'([^']+)',\s*division:\s*'([^']+)',\s*"
    r"type:\s*'([^']+)',\s*status:\s*'([^']+)',\s*max_teams:\s*(\d+)\s*\}"
)
rows = pattern.findall(raw)
print(f"Total tournois : {len(rows)}")

cats = Counter(r[8] for r in rows)  # index 8 = type
print("Types :")
for t, c in sorted(cats.items()):
    print(f"  {t:10s} × {c}")

divs = Counter(r[7] for r in rows)
print("Divisions :")
for d, c in sorted(divs.items()):
    print(f"  {d:10s} × {c}")

escape = lambda s: s.replace("'", "''")

sql = f"""\
-- ═══════════════════════════════════════════════════════════════════════════
-- MPL 2026 — Script SQL Supabase (v3)
-- Table : tournaments
-- {len(rows)} tournois officiels · Saison 10 Jan 2026 → 26 Déc 2026
-- MEN&WOMEN scindés en tournois séparés Hommes + Dames
-- Colonnes sécurisées : tournament_date, tournament_type
-- ═══════════════════════════════════════════════════════════════════════════

-- ── 1. SUPPRESSION DE L'ANCIENNE TABLE ───────────────────────────────────

DROP TABLE IF EXISTS public.tournaments CASCADE;

-- ── 2. CRÉATION DE LA TABLE ───────────────────────────────────────────────

CREATE TABLE public.tournaments (
  id                TEXT        PRIMARY KEY,
  name              TEXT        NOT NULL,
  club_id           TEXT        NOT NULL,
  club_name         TEXT        NOT NULL,
  tournament_date   DATE        NOT NULL,
  region            TEXT        NOT NULL
                                CHECK (region IN ('Nord','Ouest','Centre','Est','Sud')),
  category          TEXT        NOT NULL
                                CHECK (category IN ('M25','M50','M100','M250','M500','M1000','MIXED','U10','U12','U14')),
  division          TEXT        NOT NULL
                                CHECK (division IN ('men','women','junior','mixed')),
  tournament_type   TEXT        NOT NULL
                                CHECK (tournament_type IN ('MEN','WOMEN','JUNIOR','MIXED')),
  status            TEXT        NOT NULL DEFAULT 'upcoming'
                                CHECK (status IN ('upcoming','open','completed','cancelled')),
  max_teams         INTEGER     NOT NULL DEFAULT 16,
  teams_registered  INTEGER     NOT NULL DEFAULT 0,
  created_at        TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  updated_at        TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

-- ── 3. INDEX ──────────────────────────────────────────────────────────────

CREATE INDEX idx_tournaments_date     ON public.tournaments (tournament_date);
CREATE INDEX idx_tournaments_region   ON public.tournaments (region);
CREATE INDEX idx_tournaments_category ON public.tournaments (category);
CREATE INDEX idx_tournaments_status   ON public.tournaments (status);
CREATE INDEX idx_tournaments_division ON public.tournaments (division);
CREATE INDEX idx_tournaments_type     ON public.tournaments (tournament_type);
CREATE INDEX idx_tournaments_club_id  ON public.tournaments (club_id);

-- ── 4. ROW LEVEL SECURITY ─────────────────────────────────────────────────

ALTER TABLE public.tournaments ENABLE ROW LEVEL SECURITY;

CREATE POLICY "tournaments_select_public"
  ON public.tournaments FOR SELECT USING (true);

CREATE POLICY "tournaments_insert_admin"
  ON public.tournaments FOR INSERT
  WITH CHECK (auth.role() = 'authenticated');

CREATE POLICY "tournaments_update_admin"
  ON public.tournaments FOR UPDATE
  USING (auth.role() = 'authenticated');

CREATE POLICY "tournaments_delete_admin"
  ON public.tournaments FOR DELETE
  USING (auth.role() = 'authenticated');

-- ── 5. TRIGGER updated_at ─────────────────────────────────────────────────

CREATE OR REPLACE FUNCTION public.handle_updated_at()
RETURNS TRIGGER AS $$
BEGIN
  NEW.updated_at = NOW();
  RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_tournaments_updated_at
  BEFORE UPDATE ON public.tournaments
  FOR EACH ROW EXECUTE FUNCTION public.handle_updated_at();

-- ── 6. INSERT DES DONNÉES ({len(rows)} tournois) ──────────────────────────────

INSERT INTO public.tournaments
  (id, name, club_id, club_name, tournament_date, region, category,
   division, tournament_type, status, max_teams, teams_registered)
VALUES
"""

lines = []
for (tid, name, club_id, club_name, date, region, cat, div, typ, status, max_t) in rows:
    lines.append(
        f"  ('{escape(tid)}', '{escape(name)}', '{escape(club_id)}', '{escape(club_name)}', "
        f"'{date}', '{escape(region)}', '{escape(cat)}', '{escape(div)}', "
        f"'{escape(typ)}', '{escape(status)}', {max_t}, 0)"
    )

sql += ',\n'.join(lines)
sql += f"""

ON CONFLICT (id) DO UPDATE SET
  name             = EXCLUDED.name,
  club_id          = EXCLUDED.club_id,
  club_name        = EXCLUDED.club_name,
  tournament_date  = EXCLUDED.tournament_date,
  region           = EXCLUDED.region,
  category         = EXCLUDED.category,
  division         = EXCLUDED.division,
  tournament_type  = EXCLUDED.tournament_type,
  status           = EXCLUDED.status,
  max_teams        = EXCLUDED.max_teams,
  updated_at       = NOW();

-- ── 7. VÉRIFICATION ──────────────────────────────────────────────────────

SELECT tournament_type, division, COUNT(*) AS total
FROM public.tournaments
GROUP BY tournament_type, division
ORDER BY tournament_type, division;

SELECT category, COUNT(*) AS total,
  COUNT(*) FILTER (WHERE status = 'completed') AS termines,
  COUNT(*) FILTER (WHERE status = 'open')      AS ouverts,
  COUNT(*) FILTER (WHERE status = 'upcoming')  AS a_venir
FROM public.tournaments
GROUP BY category ORDER BY category;

SELECT COUNT(*) AS total_tournois FROM public.tournaments;
-- Résultat attendu : {len(rows)} tournois

-- ── FIN DU SCRIPT ────────────────────────────────────────────────────────
"""

with open('/workspace/mpl2026_tournaments.sql', 'w') as f:
    f.write(sql)

print(f"\n✅ SQL régénéré — {len(rows)} tournois — {len(sql.splitlines())} lignes")


Total tournois : 436
Types :
  JUNIOR     × 42
  MEN        × 223
  MIXED      × 14
  WOMEN      × 157
Divisions :
  junior     × 42
  men        × 223
  mixed      × 14
  women      × 157

✅ SQL régénéré — 436 tournois — 553 lignes
